# ABSA cho Property Review với PhoBERT + LoRA (2-Head Architecture)

Notebook này fine-tuning **PhoBERT** với **LoRA** cho bài toán **Aspect-Based Sentiment Analysis (ABSA)** trên dữ liệu đánh giá phòng trọ.

## Tổng quan Training
- **Mô hình gốc**: `vinai/phobert-base` (PhoBERT)
- **Kỹ thuật**: LoRA (Low-Rank Adaptation)
- **Các aspect**: RoomQuality, Noise, Wifi, Utilities, Parking, Security, Environment, Landlord, Location, Price
- **Nhãn sentiment**:
  - `-99`: không được đề cập (not mention)
  - `-1`: tiêu cực (negative)
  - `0`: trung tính (neutral)
  - `1`: tích cực (positive)

## Kiến trúc 2-Head
- **Header 1 - Aspect Detection**: Binary classification cho mỗi aspect (mention / not-mention). Sử dụng BCE loss.
- **Header 2 - Sentiment Classification**: 3-class (positive / negative / neutral) cho mỗi aspect. Loss chỉ tính trên các aspect được mention (gated).
- Vẫn hỗ trợ linh hoạt: nhiều **attention/pooling modes** (cls, attention_pooling, gated_fusion, conditional_attention, ...), **loss variants** (gated/weighted/focal), **LoRA settings** (r, alpha, use_ffn).

# Cài đặt thư viện cần thiết

In [1]:
!pip install --upgrade pip
!pip -q install transformers peft sentencepiece scikit-learn

# Import & DEVICE

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from datetime import datetime
import json
import math

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)


DEVICE = cuda


# Load Data

In [3]:
DATA_PATH = "/tf/workspace/DoAn1/Property_Review_ABSA_Dataset.csv"

df = pd.read_csv(DATA_PATH)
print(f"Total samples: {len(df)}")
display(df.head())

Total samples: 5000


,Num_Aspects,RoomQuality,Noise,Wifi,Utilities,Parking,Security,Environment,Landlord,Location,Price,Persona,Review_Text
0,0,-99,-99,-99,-99,-99,-99,-99,-99,-99,-99,Dân văn phòng bận rộn,Tôi vừa mới đi uống một ly cà phê và thấy thời...
1,0,-99,-99,-99,-99,-99,-99,-99,-99,-99,-99,Người viết sai chính tả (nhẹ),"Mình thấy hôm nay trời đẹp quá, thích đi dạo q..."
2,0,-99,-99,-99,-99,-99,-99,-99,-99,-99,-99,"Người dễ dãi, xuề xòa",Tôi vừa đi xem một bộ phim hay vào cuối tuần r...
3,0,-99,-99,-99,-99,-99,-99,-99,-99,-99,-99,Sinh viên Gen Z,Mình không có gì để nói về phòng trọ này vì mọ...
4,0,-99,-99,-99,-99,-99,-99,-99,-99,-99,-99,Sinh viên Gen Z,Tôi vừa xem xong một bộ phim siêu hay và cảm t...


# Constants & Preprocessing

In [4]:
TEXT_COL = "Review_Text"

ASPECTS = [
    "RoomQuality", "Noise", "Wifi", "Utilities", "Parking",
    "Security", "Environment", "Landlord", "Location", "Price"
]

NUM_ASPECTS = len(ASPECTS)
OUTPUT_DIM = 4 * NUM_ASPECTS  # legacy single-head (4 flags per aspect)
ASPECT_DIM = NUM_ASPECTS       # for 2-head: aspect detection
SENTIMENT_DIM = 3 * NUM_ASPECTS  # for 2-head: pos/neg/neu per aspect

print(f"NUM_ASPECTS = {NUM_ASPECTS}")
print(f"OUTPUT_DIM (legacy) = {OUTPUT_DIM}")
print(f"ASPECT_DIM = {ASPECT_DIM}, SENTIMENT_DIM = {SENTIMENT_DIM}")

NUM_ASPECTS = 10
OUTPUT_DIM (legacy) = 40
ASPECT_DIM = 10, SENTIMENT_DIM = 30


In [5]:
def create_absa_vector_display_columns(absa_vector, aspects_list):
    col_data = {}
    current_index = 0
    for aspect in aspects_list:
        mention = int(absa_vector[current_index])
        pos = int(absa_vector[current_index + 1])
        neg = int(absa_vector[current_index + 2])
        neu = int(absa_vector[current_index + 3])

        col_data[f"m_{aspect}"] = mention

        sentiment_label = -99
        if mention == 1:
            if pos == 1:
                sentiment_label = 1
            elif neg == 1:
                sentiment_label = -1
            elif neu == 1:
                sentiment_label = 0

        col_data[f"y_{aspect}"] = sentiment_label

        current_index += 4
    return pd.Series(col_data)

In [6]:
def build_absa_vector(row):
    labels = []
    for aspect in ASPECTS:
        raw = int(row[aspect])
        mention = 1 if raw != -99 else 0
        pos = 1 if raw == 1 else 0
        neg = 1 if raw == -1 else 0
        neu = 1 if raw == 0 else 0
        labels.extend([mention, pos, neg, neu])
    return np.array(labels, dtype=np.float32)

df["absa_vector"] = df.apply(build_absa_vector, axis=1)

decoded_columns = df["absa_vector"].apply(
    lambda x: create_absa_vector_display_columns(x, ASPECTS)
)

columns_to_display = [TEXT_COL] + ASPECTS + [col for col in decoded_columns.columns]
display_df = pd.concat([df[TEXT_COL], df[ASPECTS], decoded_columns], axis=1)
display(display_df[columns_to_display].head())

,Review_Text,RoomQuality,Noise,Wifi,Utilities,Parking,Security,Environment,Landlord,Location,...,m_Security,y_Security,m_Environment,y_Environment,m_Landlord,y_Landlord,m_Location,y_Location,m_Price,y_Price
0,Tôi vừa mới đi uống một ly cà phê và thấy thời...,-99,-99,-99,-99,-99,-99,-99,-99,-99,...,0,-99,0,-99,0,-99,0,-99,0,-99
1,"Mình thấy hôm nay trời đẹp quá, thích đi dạo q...",-99,-99,-99,-99,-99,-99,-99,-99,-99,...,0,-99,0,-99,0,-99,0,-99,0,-99
2,Tôi vừa đi xem một bộ phim hay vào cuối tuần r...,-99,-99,-99,-99,-99,-99,-99,-99,-99,...,0,-99,0,-99,0,-99,0,-99,0,-99
3,Mình không có gì để nói về phòng trọ này vì mọ...,-99,-99,-99,-99,-99,-99,-99,-99,-99,...,0,-99,0,-99,0,-99,0,-99,0,-99
4,Tôi vừa xem xong một bộ phim siêu hay và cảm t...,-99,-99,-99,-99,-99,-99,-99,-99,-99,...,0,-99,0,-99,0,-99,0,-99,0,-99


# Two-Head Label Builder (Aspect Detection + Sentiment)

In [7]:
def build_twohead_labels(row):
    """Build separate labels for 2-head architecture."""
    mentions = []
    sentiments = []  # 3 values (pos, neg, neu) per aspect
    for aspect in ASPECTS:
        raw = int(row[aspect])
        mention = 1 if raw != -99 else 0
        pos = 1 if raw == 1 else 0
        neg = 1 if raw == -1 else 0
        neu = 1 if raw == 0 else 0
        mentions.append(float(mention))
        sentiments.extend([float(pos), float(neg), float(neu)])
    return np.array(mentions, dtype=np.float32), np.array(sentiments, dtype=np.float32)

df["aspect_labels"], df["sentiment_labels"] = zip(*df.apply(build_twohead_labels, axis=1))

# Quick preview
print("Aspect labels shape example:", df["aspect_labels"].iloc[0].shape)
print("Sentiment labels shape example:", df["sentiment_labels"].iloc[0].shape)
print("Sample aspect_labels[0]:", df["aspect_labels"].iloc[0])
print("Sample sentiment_labels[0]:", df["sentiment_labels"].iloc[0])

Aspect labels shape example: (10,)
Sentiment labels shape example: (30,)
Sample aspect_labels[0]: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Sample sentiment_labels[0]: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0.]


# Train / Validation / Test Split

In [8]:
# Improved train/val/test split with MULTILABEL stratification on each aspect
# This ensures rare aspects (e.g. "Landlord") are distributed across splits instead of being
# concentrated in train only.

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    USE_MLSS = True
except ImportError:
    print("WARNING: iterative-stratification not installed. Falling back to Num_Aspects>0 stratification.")
    print("Run in Colab: !pip install iterative-stratification")
    USE_MLSS = False

if USE_MLSS:
    aspect_label_matrix = np.stack(df["aspect_labels"].values)   # shape (N, 10) multi-label binary

    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    train_idx, temp_idx = next(msss.split(df.index, aspect_label_matrix))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    temp_df  = df.iloc[temp_idx].reset_index(drop=True)

    temp_aspect = np.stack(temp_df["aspect_labels"].values)
    msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
    val_rel, test_rel = next(msss2.split(temp_df.index, temp_aspect))

    val_df  = temp_df.iloc[val_rel].reset_index(drop=True)
    test_df = temp_df.iloc[test_rel].reset_index(drop=True)
else:
    # Fallback (old behavior)
    train_df, temp_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df["Num_Aspects"] > 0
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=42, stratify=temp_df["Num_Aspects"] > 0
    )

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
# Optional: verify per-aspect distribution
if USE_MLSS:
    for i, asp in enumerate(ASPECTS):
        tr = train_df["aspect_labels"].apply(lambda x: x[i]).sum()
        va = val_df["aspect_labels"].apply(lambda x: x[i]).sum()
        te = test_df["aspect_labels"].apply(lambda x: x[i]).sum()
        print(f"  {asp:12s}: train={int(tr):4d}  val={int(va):3d}  test={int(te):3d}")


Run in Colab: !pip install iterative-stratification
Train: 4000 | Val: 500 | Test: 500


# Tokenizer & Dataset

In [9]:
MODEL_NAME = "vinai/phobert-base"
MAX_LEN = 256
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class ABSADataset(Dataset):
    def __init__(self, df):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = np.stack(df["absa_vector"].values)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

train_loader = DataLoader(ABSADataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(ABSADataset(val_df),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(ABSADataset(test_df),  batch_size=BATCH_SIZE, shuffle=False)

# ABSATwoHeadDataset (cho kiến trúc 2 header)

In [10]:
class ABSATwoHeadDataset(Dataset):
    """Dataset cho 2-head: aspect detection + sentiment classification."""
    def __init__(self, df):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.aspect_labels = np.stack(df["aspect_labels"].values)
        self.sentiment_labels = np.stack(df["sentiment_labels"].values)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "aspect_labels": torch.tensor(self.aspect_labels[idx], dtype=torch.float),
            "sentiment_labels": torch.tensor(self.sentiment_labels[idx], dtype=torch.float)
        }

# Tạo DataLoader cho 2-head (chỉ dùng trong Nhóm 3)
train_loader_2h = DataLoader(ABSATwoHeadDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader_2h   = DataLoader(ABSATwoHeadDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False)
test_loader_2h  = DataLoader(ABSATwoHeadDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False)
print("Two-head loaders created:", len(train_loader_2h.dataset), len(val_loader_2h.dataset), len(test_loader_2h.dataset))

Two-head loaders created: 4000 500 500


# ABSAFeatureExtractor

In [11]:
class ABSAFeatureExtractor(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        mode: str = "gated_fusion",
        num_aspects: int = None,
        dropout: float = 0.1,
        attention_dropout: float = 0.1,
        num_heads: int = 4,
        per_aspect: bool = False
    ):
        super().__init__()

        self.temperature = nn.Parameter(torch.tensor(1.0))
        self.mode = mode.lower()
        self.hidden_size = hidden_size
        self.per_aspect = per_aspect

        self.dropout = nn.Dropout(dropout)
        self.attn_dropout = nn.Dropout(attention_dropout)
        self.norm = nn.LayerNorm(hidden_size)

        if self.mode in ["attention_pooling", "gated_fusion", "conditional_attention"]:
            self.attn_score = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.Tanh(),
                nn.Linear(hidden_size, 1)
            )

        if self.mode == "gated_fusion":
            self.gate = nn.Sequential(
                nn.Linear(hidden_size * 2, hidden_size),
                nn.GELU(),
                nn.Linear(hidden_size, 1)
            )

        if self.mode == "conditional_attention":
            if num_aspects is None:
                raise ValueError("num_aspects must be provided for conditional_attention")
            self.aspect_queries = nn.Parameter(torch.randn(num_aspects, hidden_size))
            nn.init.xavier_uniform_(self.aspect_queries)

        if self.mode == "multihead_attention_pooling":
            self.multihead_attn = nn.MultiheadAttention(
                embed_dim=hidden_size, num_heads=num_heads,
                dropout=attention_dropout, batch_first=True
            )
            self.mha_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, last_hidden_state, attention_mask=None):
        if self.mode == "cls":
            return self.dropout(last_hidden_state[:, 0, :])

        elif self.mode == "attention_pooling":
            scores = self.attn_score(last_hidden_state)
            if attention_mask is not None:
                mask = attention_mask.unsqueeze(-1)
                scores = scores.masked_fill(mask == 0, -1e9)
            weights = torch.softmax(scores / self.temperature.abs().clamp(min=0.5), dim=1)
            pooled = (weights * last_hidden_state).sum(dim=1)
            return self.dropout(pooled)

        elif self.mode == "gated_fusion":
            cls_vec = last_hidden_state[:, 0, :]
            scores = self.attn_score(last_hidden_state)
            if attention_mask is not None:
                mask = attention_mask.unsqueeze(-1)
                scores = scores.masked_fill(mask == 0, -1e9)
            weights = torch.softmax(scores / self.temperature.abs().clamp(min=0.5), dim=1)
            attn_pooled = (weights * last_hidden_state).sum(dim=1)
            gate = torch.sigmoid(self.gate(torch.cat([cls_vec, attn_pooled], dim=-1)))
            fused = gate * cls_vec + (1 - gate) * attn_pooled
            return self.dropout(fused)

        elif self.mode == "conditional_attention":
            if attention_mask is not None:
                mask = attention_mask.unsqueeze(-1)
            else:
                mask = torch.ones(
                    last_hidden_state.shape[0], 1, 1,
                    device=last_hidden_state.device
                )
            queries = self.aspect_queries.unsqueeze(0)
            keys = last_hidden_state.unsqueeze(1)
            scores = torch.matmul(queries, keys.transpose(-2, -1)) / math.sqrt(self.hidden_size)
            scores = scores.squeeze(2)
            scores = scores.masked_fill(mask.transpose(1, 2) == 0, -1e9)
            weights = torch.softmax(scores, dim=-1)
            aspect_vectors = torch.matmul(weights, last_hidden_state)
            if self.per_aspect:
                # Return per-aspect vectors for sentiment header (B, N, H)
                return self.dropout(aspect_vectors)
            pooled = aspect_vectors.mean(dim=1)
            return self.dropout(pooled)

        elif self.mode == "multihead_attention_pooling":
            attn_out, _ = self.multihead_attn(
                last_hidden_state, last_hidden_state, last_hidden_state,
                key_padding_mask=(attention_mask == 0) if attention_mask is not None else None
            )
            pooled = self.mha_proj(attn_out.mean(dim=1))
            return self.dropout(pooled)

        else:
            raise ValueError(f"Unknown mode: {self.mode}")

# ABSALoss

In [12]:
class ABSALoss(nn.Module):
    def __init__(
        self,
        num_aspects: int,
        use_gated=True,
        use_weighted=True,
        use_focal=True,
        gamma=2.0,
        alpha=0.25,
        mention_weight=1.2,
        sentiment_weight=0.8,
        absent_sentiment_scale=0.15,
        label_smoothing=0.02
    ):
        super().__init__()
        self.num_aspects = num_aspects
        self.use_gated = use_gated
        self.use_weighted = use_weighted
        self.use_focal = use_focal
        self.gamma = gamma
        self.alpha = alpha
        self.mention_weight = mention_weight
        self.sentiment_weight = sentiment_weight
        self.absent_sentiment_scale = absent_sentiment_scale
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, labels: torch.Tensor):
        device = logits.device

        if self.label_smoothing > 0:
            labels = (
                labels * (1 - self.label_smoothing)
                + 0.5 * self.label_smoothing
            )

        bce_loss = F.binary_cross_entropy_with_logits(
            logits, labels, reduction='none'
        )

        if self.use_gated:
            batch_size = logits.shape[0]
            mention_mask = labels[:, 0::4]
            mention_mask = mention_mask.unsqueeze(-1)
            mention_mask = mention_mask.repeat(1, 1, 4)
            mention_mask = mention_mask.reshape(batch_size, -1)

            mention_mask[:, 0::4] = 1.0
            bce_loss = bce_loss * mention_mask

        if self.use_weighted:
            batch_size = logits.shape[0]
            weights = torch.ones_like(bce_loss)
            for i in range(self.num_aspects):
                start = i * 4
                weights[:, start] = self.mention_weight
                weights[:, start+1:start+4] = self.sentiment_weight
            bce_loss = bce_loss * weights

        if self.use_focal:
            probs = torch.sigmoid(logits)
            pt = torch.where(labels > 0.5, probs, 1 - probs)
            focal_weight = (1 - pt) ** self.gamma
            if self.alpha > 0:
                alpha_weight = torch.where(
                    labels > 0.5, self.alpha, 1 - self.alpha
                )
                focal_weight = focal_weight * alpha_weight
            bce_loss = bce_loss * focal_weight

        return bce_loss.mean()

# ABSATwoHeadLoss (hỗ trợ gated sentiment loss + focal + weighted)

In [13]:
class ABSATwoHeadLoss(nn.Module):
    """Loss cho 2-head architecture.
    - Aspect: BCE (có thể focal/weighted)
    - Sentiment: BCE gated chỉ trên aspect được mention (hỗ trợ focal/weighted)
    """
    def __init__(
        self,
        num_aspects: int,
        use_gated: bool = True,
        use_weighted: bool = True,
        use_focal: bool = True,
        gamma: float = 2.0,
        alpha: float = 0.25,
        aspect_weight: float = 1.0,
        sentiment_weight: float = 0.8,
        label_smoothing: float = 0.02
    ):
        super().__init__()
        self.num_aspects = num_aspects
        self.use_gated = use_gated
        self.use_weighted = use_weighted
        self.use_focal = use_focal
        self.gamma = gamma
        self.alpha = alpha
        self.aspect_weight = aspect_weight
        self.sentiment_weight = sentiment_weight
        self.label_smoothing = label_smoothing

    def forward(self, aspect_logits: torch.Tensor, sentiment_logits: torch.Tensor,
                aspect_labels: torch.Tensor, sentiment_labels: torch.Tensor):
        device = aspect_logits.device
        batch_size = aspect_logits.shape[0]

        # Optional label smoothing on targets
        if self.label_smoothing > 0:
            aspect_labels = aspect_labels * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
            sentiment_labels = sentiment_labels * (1 - self.label_smoothing) + 0.5 * self.label_smoothing

        # ===== Aspect Detection Loss (BCE) =====
        aspect_bce = F.binary_cross_entropy_with_logits(aspect_logits, aspect_labels, reduction="none")
        aspect_loss = aspect_bce.mean()
        if self.use_weighted:
            aspect_loss = aspect_loss * self.aspect_weight
        if self.use_focal:
            probs = torch.sigmoid(aspect_logits)
            pt = torch.where(aspect_labels > 0.5, probs, 1 - probs)
            focal_w = (1 - pt) ** self.gamma
            if self.alpha > 0:
                focal_w = focal_w * torch.where(aspect_labels > 0.5, self.alpha, 1 - self.alpha)
            aspect_loss = (aspect_bce * focal_w).mean() * (self.aspect_weight if self.use_weighted else 1.0)

        # ===== Sentiment Loss (BCE, gated by mention) =====
        sent_bce = F.binary_cross_entropy_with_logits(sentiment_logits, sentiment_labels, reduction="none")
        if self.use_gated:
            # mask: repeat aspect mention across 3 sentiment slots
            mention_mask = aspect_labels.unsqueeze(-1).repeat(1, 1, 3).reshape(batch_size, -1)
            sent_bce = sent_bce * mention_mask

        sent_loss = sent_bce.mean()
        if self.use_weighted:
            # apply per-sentiment weight (scale all 3 equally per aspect)
            sent_loss = sent_loss * self.sentiment_weight
        if self.use_focal:
            probs = torch.sigmoid(sentiment_logits)
            pt = torch.where(sentiment_labels > 0.5, probs, 1 - probs)
            focal_w = (1 - pt) ** self.gamma
            if self.alpha > 0:
                focal_w = focal_w * torch.where(sentiment_labels > 0.5, self.alpha, 1 - self.alpha)
            gated_sent_bce = sent_bce * focal_w
            sent_loss = gated_sent_bce.mean() * (self.sentiment_weight if self.use_weighted else 1.0)

        total = aspect_loss + sent_loss
        return total, aspect_loss.detach(), sent_loss.detach()

# build_lora_encoder

In [14]:
def build_lora_encoder(
    model_name: str,
    r: int = 16,
    lora_alpha: int = 32,
    lora_dropout: float = 0.05,
    use_ffn: bool = False,
) -> PeftModel:
    base_model = AutoModel.from_pretrained(model_name)

    target_modules = ["query", "key", "value"]
    if use_ffn:
        target_modules.extend(["dense"])

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        bias="none",
        inference_mode=False,
    )

    peft_model = get_peft_model(base_model, lora_config)
    print(f"LoRA | r={r}, alpha={lora_alpha}, use_ffn={use_ffn}")
    peft_model.print_trainable_parameters()
    return peft_model

# Models

In [15]:
class LoRABERTABSA_TwoHead(nn.Module):
    """2-Head ABSA model với cùng một cơ chế attention (feature extractor) cho cả 2 header.
    - feature_mode: chung cho cả aspect detection và sentiment classification.
    - Hỗ trợ: cls, attention_pooling, gated_fusion, conditional_attention, multihead_attention_pooling
    """
    def __init__(
        self,
        model_name: str,
        feature_mode: str = "gated_fusion",
        lora_r: int = 32,
        lora_alpha: int = 64,
        lora_dropout: float = 0.05,
        use_ffn: bool = True,
        dropout: float = 0.1,
        num_aspects: int = None,
    ):
        super().__init__()
        if num_aspects is None:
            num_aspects = NUM_ASPECTS
        self.num_aspects = num_aspects
        self.feature_mode = feature_mode

        self.encoder = build_lora_encoder(
            model_name, r=lora_r, lora_alpha=lora_alpha,
            lora_dropout=lora_dropout, use_ffn=use_ffn
        )
        hidden_size = self.encoder.config.hidden_size

        # Dùng chung một feature extractor cho cả 2 header
        # (đảm bảo cùng cơ chế attention/pooling)
        per_aspect = False  # luôn dùng shared representation khi cùng mode
        self.feature_extractor = ABSAFeatureExtractor(
            hidden_size=hidden_size,
            mode=feature_mode,
            num_aspects=num_aspects,
            dropout=dropout,
            per_aspect=per_aspect
        )

        self.aspect_head = nn.Linear(hidden_size, num_aspects)
        self.sentiment_head = nn.Linear(hidden_size, num_aspects * 3)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state

        feat = self.feature_extractor(last_hidden, attention_mask=attention_mask)
        feat = self.dropout(feat)

        if feat.dim() == 3:
            # === Dành cho các mode trả về 3D như 'conditional_attention' ===
            # feat shape: [B, N, H] (Batch, Num_Aspects, Hidden_Size)
            B, N, H = feat.shape

            # 1. Aspect Logits
            a_logits_3d = self.aspect_head(feat) # shape: [B, N, N]
            # Lấy đường chéo: Dùng đặc trưng của aspect i để dự đoán cho aspect i -> shape: [B, N]
            aspect_logits = torch.diagonal(a_logits_3d, dim1=1, dim2=2)

            # 2. Sentiment Logits
            s_logits_3d = self.sentiment_head(feat) # shape: [B, N, N*3]
            # Tách 3 class sentiment ra: shape: [B, N, N, 3]
            s_logits_4d = s_logits_3d.view(B, N, N, 3)
            # Lấy đường chéo: Dùng đặc trưng aspect i cho cảm xúc aspect i -> shape: [B, 3, N]
            s_logits_diag = torch.diagonal(s_logits_4d, dim1=1, dim2=2)
            # Xoay và làm phẳng lại thành [B, N*3] để khớp hoàn toàn với Loss Function
            sentiment_logits = s_logits_diag.transpose(1, 2).reshape(B, N * 3)

        else:
            # === Dành cho các mode trả về 2D như 'gated_fusion', 'cls' ===
            # feat shape: [B, H] (Batch, Hidden_Size)
            aspect_logits = self.aspect_head(feat)           # (B, N)
            sentiment_logits = self.sentiment_head(feat)     # (B, 3N)

        return aspect_logits, sentiment_logits

# Two-Head: Evaluate, Save, Train helpers

In [16]:
def save_twohead_model(model, save_dir: str, model_name: str = "twohead"):
    os.makedirs(save_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.join(save_dir, f"{model_name}_{timestamp}")
    os.makedirs(save_path, exist_ok=True)

    if hasattr(model, "encoder") and hasattr(model.encoder, "save_pretrained"):
        model.encoder.save_pretrained(os.path.join(save_path, "lora_adapter"))
        torch.save(model.aspect_head.state_dict(), os.path.join(save_path, "aspect_head.pt"))
        torch.save(model.sentiment_head.state_dict(), os.path.join(save_path, "sentiment_head.pt"))
        cfg = {
            "feature_mode": getattr(model, "feature_mode", None),
            "num_aspects": model.num_aspects,
        }
        with open(os.path.join(save_path, "config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Saved TwoHead LoRA model (shared mode={cfg['feature_mode']}) to {save_path}")
    else:
        torch.save(model.state_dict(), os.path.join(save_path, "model.pth"))
        print(f"Saved TwoHead model to {save_path}")
    return save_path

@torch.no_grad()
@torch.no_grad()
def evaluate_twohead(model, data_loader, aspects=None, threshold=0.5):
    """Quick two-head eval.

    Returns:
        aspect_f1, gt_sentiment_f1 (gated on ground-truth mentions),
        pipeline_sentiment_f1 (gated on predicted mentions - what user sees at inference),
        combined (aspect + pipeline_sentiment averaged)
    """
    if aspects is None:
        aspects = ASPECTS
    model.eval()
    all_a_t, all_a_p = [], []
    gt_sent_t, gt_sent_p = [], []
    pipe_sent_t, pipe_sent_p = [], []

    for batch in tqdm(data_loader, desc="Eval twohead", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        a_labels = batch["aspect_labels"].cpu().numpy()
        s_labels = batch["sentiment_labels"].cpu().numpy()

        a_logits, s_logits = model(input_ids, attention_mask)
        a_probs = torch.sigmoid(a_logits).cpu().numpy()
        s_probs = torch.sigmoid(s_logits).cpu().numpy()
        a_pred = (a_probs > threshold).astype(int)
        s_pred = (s_probs > threshold).astype(int)

        all_a_t.append(a_labels)
        all_a_p.append(a_pred)

        for b in range(a_labels.shape[0]):
            for i in range(len(aspects)):
                st = i * 3
                # GT-gated (current "pure" sentiment head strength)
                if a_labels[b, i] > 0.5:
                    gt_sent_t.append(s_labels[b, st:st+3])
                    gt_sent_p.append(s_pred[b, st:st+3])
                # Pipeline / End-to-End (realistic inference gate)
                if a_pred[b, i] > 0.5:
                    pipe_sent_t.append(s_labels[b, st:st+3])
                    pipe_sent_p.append(s_pred[b, st:st+3])

    a_true = np.concatenate(all_a_t, axis=0)
    a_pred = np.concatenate(all_a_p, axis=0)
    aspect_f1 = f1_score(a_true, a_pred, average="macro", zero_division=0)

    # GT gated sentiment
    if gt_sent_t:
        gt_sent_f1 = f1_score(np.stack(gt_sent_t), np.stack(gt_sent_p), average="macro", zero_division=0)
    else:
        gt_sent_f1 = 0.0

    # Pipeline (pred-gated) sentiment
    if pipe_sent_t:
        pipe_sent_f1 = f1_score(np.stack(pipe_sent_t), np.stack(pipe_sent_p), average="macro", zero_division=0)
    else:
        pipe_sent_f1 = 0.0

    combined = (aspect_f1 + pipe_sent_f1) / 2.0
    return float(aspect_f1), float(gt_sent_f1), float(pipe_sent_f1), float(combined)


def evaluate_twohead_detailed(model, data_loader, aspects=None, verbose=True):
    """Detailed two-head evaluation.

    Reports BOTH:
      - Gated Sentiment F1 (oracle: only on ground-truth mentioned aspects)
      - Pipeline Sentiment F1 (realistic: gated on predicted aspects)
    """
    if aspects is None:
        aspects = ASPECTS
    model.eval()
    all_aspect_true, all_aspect_pred = [], []
    per_asp_gt_true = {asp: [] for asp in aspects}
    per_asp_gt_pred = {asp: [] for asp in aspects}
    per_asp_pipe_true = {asp: [] for asp in aspects}
    per_asp_pipe_pred = {asp: [] for asp in aspects}

    for batch in tqdm(data_loader, desc="Evaluating twohead detailed", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        a_labels = batch["aspect_labels"].cpu().numpy()
        s_labels = batch["sentiment_labels"].cpu().numpy()

        a_logits, s_logits = model(input_ids, attention_mask)
        a_pred = (torch.sigmoid(a_logits) > 0.5).int().cpu().numpy()
        s_pred = (torch.sigmoid(s_logits) > 0.5).int().cpu().numpy()

        all_aspect_true.append(a_labels)
        all_aspect_pred.append(a_pred)

        for b in range(a_labels.shape[0]):
            for i, asp in enumerate(aspects):
                st = i * 3
                # GT gate
                if a_labels[b, i] > 0.5:
                    per_asp_gt_true[asp].append(s_labels[b, st:st+3])
                    per_asp_gt_pred[asp].append(s_pred[b, st:st+3])
                # Pred gate (pipeline)
                if a_pred[b, i] > 0.5:
                    per_asp_pipe_true[asp].append(s_labels[b, st:st+3])
                    per_asp_pipe_pred[asp].append(s_pred[b, st:st+3])

    a_true = np.concatenate(all_aspect_true, axis=0)
    a_pred = np.concatenate(all_aspect_pred, axis=0)
    overall_mention_f1 = f1_score(a_true, a_pred, average="macro", zero_division=0)

    # GT-gated overall
    gt_all_t, gt_all_p = [], []
    for asp in aspects:
        if per_asp_gt_true[asp]:
            gt_all_t.append(np.stack(per_asp_gt_true[asp]))
            gt_all_p.append(np.stack(per_asp_gt_pred[asp]))
    overall_gt_sent = f1_score(np.concatenate(gt_all_t), np.concatenate(gt_all_p), average="macro", zero_division=0) if gt_all_t else 0.0

    # Pipeline overall
    pipe_all_t, pipe_all_p = [], []
    for asp in aspects:
        if per_asp_pipe_true[asp]:
            pipe_all_t.append(np.stack(per_asp_pipe_true[asp]))
            pipe_all_p.append(np.stack(per_asp_pipe_pred[asp]))
    overall_pipe_sent = f1_score(np.concatenate(pipe_all_t), np.concatenate(pipe_all_p), average="macro", zero_division=0) if pipe_all_t else 0.0

    per_aspect = {}
    mention_f1s = []
    gt_sent_f1s = []
    pipe_sent_f1s = []

    for i, asp in enumerate(aspects):
        m_f1 = f1_score(a_true[:, i], a_pred[:, i], average="binary", zero_division=0)
        mention_f1s.append(m_f1)

        # GT sentiment for this aspect
        if per_asp_gt_true[asp]:
            s_f1 = f1_score(np.stack(per_asp_gt_true[asp]), np.stack(per_asp_gt_pred[asp]), average="macro", zero_division=0)
        else:
            s_f1 = 0.0
        gt_sent_f1s.append(s_f1)

        # Pipeline sentiment for this aspect
        if per_asp_pipe_true[asp]:
            p_f1 = f1_score(np.stack(per_asp_pipe_true[asp]), np.stack(per_asp_pipe_pred[asp]), average="macro", zero_division=0)
        else:
            p_f1 = 0.0
        pipe_sent_f1s.append(p_f1)

        per_aspect[asp] = {
            "mention_f1": round(m_f1, 4),
            "sentiment_f1": round(s_f1, 4),           # old name = GT-gated
            "pipeline_sentiment_f1": round(p_f1, 4)   # new: what inference will actually produce
        }

    macro_mention = float(np.mean(mention_f1s))
    macro_gt_sent = float(np.mean([s for s in gt_sent_f1s if s > 0]) or 0.0)
    macro_pipe_sent = float(np.mean([s for s in pipe_sent_f1s if s > 0]) or 0.0)

    result = {
        "overall_mention_f1": round(float(overall_mention_f1), 4),
        "overall_sentiment_f1": round(float(overall_gt_sent), 4),           # GT-gated (for head analysis)
        "overall_pipeline_sentiment_f1": round(float(overall_pipe_sent), 4), # Realistic end-to-end
        "macro_per_aspect_mention_f1": round(macro_mention, 4),
        "macro_per_aspect_sentiment_f1": round(macro_gt_sent, 4),
        "macro_per_aspect_pipeline_sentiment_f1": round(macro_pipe_sent, 4),
        "per_aspect": per_aspect
    }

    if verbose:
        print("\n===== TWOHEAD DETAILED EVAL (with PIPELINE) =====")
        print(f"Overall Mention F1                  : {result['overall_mention_f1']:.4f}")
        print(f"Overall Sentiment F1 (GT-gated)     : {result['overall_sentiment_f1']:.4f}")
        print(f"Overall Sentiment F1 (PIPELINE)     : {result['overall_pipeline_sentiment_f1']:.4f}  <-- what users see")
        print("Per Aspect (mention | GT-sent | PIPELINE-sent):")
        for asp, m in per_aspect.items():
            print(f"  {asp:12s}: {m['mention_f1']:.4f} | {m['sentiment_f1']:.4f} | {m['pipeline_sentiment_f1']:.4f}")

    return result


def train_twohead_model(model, train_loader, val_loader, epochs=5, lr=2e-5,
                        model_name_for_log="TwoHead", loss_fn=None,
                        save_dir: str = None,
                        patience: int = None,
                        monitor: str = "val_combined",
                        mode: str = "max",
                        min_delta: float = 0.0,
                        # LR Scheduler
                        warmup_ratio: float = 0.1,
                        use_scheduler: bool = True,
                        # Best checkpoint
                        save_best: bool = True):
    """Train loop for two-head model with:
      - Linear Warmup + Decay Scheduler (standard for PhoBERT)
      - Early Stopping (patience / monitor)
      - Automatic BEST checkpoint saving + reload best weights into returned model

    The returned model and path will point to the BEST checkpoint when save_best=True.
    """
    model = model.to(DEVICE)
    if loss_fn is None:
        loss_fn = ABSATwoHeadLoss(num_aspects=getattr(model, "num_aspects", NUM_ASPECTS))

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )

    # Scheduler (per-step) - very important for stable Transformer fine-tuning
    scheduler = None
    if use_scheduler:
        total_steps = epochs * len(train_loader)
        warmup_steps = int(total_steps * warmup_ratio)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
        print(f"[Scheduler] Linear Warmup+Decay enabled | warmup={warmup_steps}/{total_steps} steps")

    history = []
    best_score = None
    best_epoch = None
    no_improve_count = 0
    best_save_path = None

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_a_loss = 0.0
        total_s_loss = 0.0

        for batch in tqdm(train_loader, desc=f"{model_name_for_log} | Epoch {epoch}"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            a_labels = batch["aspect_labels"].to(DEVICE)
            s_labels = batch["sentiment_labels"].to(DEVICE)

            a_logits, s_logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss, a_l, s_l = loss_fn(a_logits, s_logits, a_labels, s_labels)

            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()
            total_a_loss += a_l.item() if hasattr(a_l, "item") else float(a_l)
            total_s_loss += s_l.item() if hasattr(s_l, "item") else float(s_l)

        avg_loss = total_loss / max(1, len(train_loader))
        avg_a = total_a_loss / max(1, len(train_loader))
        avg_s = total_s_loss / max(1, len(train_loader))

        # 4-value evaluate (aspect, gt_sent, pipeline_sent, combined)
        va, vs_gt, vs_pipe, vc = evaluate_twohead(model, val_loader)

        history.append({
            "epoch": epoch,
            "loss": round(avg_loss, 4),
            "aspect_loss": round(avg_a, 4),
            "sent_loss": round(avg_s, 4),
            "val_aspect_f1": round(va, 4),
            "val_gt_sent_f1": round(vs_gt, 4),
            "val_pipeline_sent_f1": round(vs_pipe, 4),
            "val_combined": round(vc, 4),
            "lr": optimizer.param_groups[0]["lr"]
        })

        # Choose score for early stopping / best tracking
        score_map = {
            "val_aspect_f1": va,
            "val_sent_f1": vs_gt,
            "val_pipeline_sent_f1": vs_pipe,
            "val_combined": vc
        }
        current_score = score_map.get(monitor, vc)

        improved = False
        if best_score is None:
            improved = True
        else:
            if mode == "max":
                if current_score >= best_score + min_delta:
                    improved = True
            else:
                if current_score <= best_score - min_delta:
                    improved = True

        if improved:
            best_score = current_score
            best_epoch = epoch
            no_improve_count = 0

            # === SAVE BEST CHECKPOINT ===
            if save_best:
                best_name = f"{model_name_for_log}_BEST"
                best_save_path = save_twohead_model(model, save_dir or "./models", best_name)
                print(f"  ★ New BEST ({monitor}={best_score:.4f}) saved → {best_save_path}")
        else:
            no_improve_count += 1

        print(f"{model_name_for_log} | Epoch {epoch} | Loss={avg_loss:.4f} | "
              f"Aspect={va:.4f} GT-Sent={vs_gt:.4f} Pipe-Sent={vs_pipe:.4f} Comb={vc:.4f} | "
              f"Best {monitor}={best_score:.4f} @ep{best_epoch}")

        if patience is not None and patience > 0 and no_improve_count >= patience:
            print(f"\n>>> Early stopping triggered at epoch {epoch}.")
            print(f">>> Best {monitor} = {best_score:.4f} at epoch {best_epoch}")
            break

    # === Finalize: prefer returning the BEST model & path ===
    final_dir = save_dir or "./models"

    if save_best and best_save_path is not None:
        print(f"\n[Best Checkpoint] Loading BEST weights into returned model from: {best_save_path}")
        # Use the existing robust loader so the returned model object has best weights
        try:
            model = load_twohead_model(best_save_path)
            returned_path = best_save_path
        except Exception as e:
            print(f"Warning: could not reload best via load_twohead_model ({e}). "
                  f"Returning model at stop time. Best folder still available at: {best_save_path}")
            returned_path = best_save_path
    else:
        # No best saving or no improvement happened → save the final state
        returned_path = save_twohead_model(model, final_dir, model_name_for_log)

    # Always save a "last" / final version for reference (in addition to best)
    try:
        last_path = save_twohead_model(model, final_dir, f"{model_name_for_log}_LAST")
        print(f"Final (last) model also saved at: {last_path}")
    except Exception:
        pass

    try:
        with open(os.path.join(final_dir, "history.json"), "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)
    except Exception as e:
        print("Warning: could not save history.json:", e)

    if best_epoch is not None:
        print(f"\nBest {monitor} achieved: {best_score:.4f} at epoch {best_epoch}")
    print(f"Returning model from: {returned_path}\n")

    return model, history, returned_path



# Training

Cả **2 header** (aspect detection + sentiment classification) dùng **cùng một feature_mode** (cùng cơ chế attention/pooling).

Các biến thể attention được thử nghiệm (chung cho cả hai header):
- gated_fusion
- conditional_attention
- attention_pooling (kết hợp LoRA rank cao hơn)

Kết hợp với LoRA mạnh (r=32/40, alpha cao, use_ffn=True) + full loss (gated + weighted + focal).


## GatedFusion


In [ ]:
print("=== NHÓM 1: gated_fusion | FullLoss | LoRA(r=32,a=64,ffn) ===")

loss_fn_3 = ABSATwoHeadLoss(
    num_aspects=NUM_ASPECTS,
    use_gated=True,
    use_weighted=True,
    use_focal=True,
    gamma=2.0,
    aspect_weight=1.0,
    sentiment_weight=0.9
)

model_3_1 = LoRABERTABSA_TwoHead(
    model_name=MODEL_NAME,
    feature_mode="gated_fusion",
    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    use_ffn=True,
    dropout=0.1
).to(DEVICE)

trained_3_1, hist_3_1, path_3_1 = train_twohead_model(
    model=model_3_1,
    train_loader=train_loader_2h,
    val_loader=val_loader_2h,
    epochs=30,
    lr=2e-4,
    model_name_for_log="TwoHead_Shared_GatedFusion",
    loss_fn=loss_fn_3,
    save_dir="./models",
    patience=5,
    monitor="val_pipeline_sent_f1",
    mode="max",
    min_delta=0.0015,
    warmup_ratio=0.08,
    use_scheduler=True,
    save_best=True
)

a_f1, s_f1_gt, s_f1_pipe, comb = evaluate_twohead(trained_3_1, test_loader_2h)
detail_3_1 = evaluate_twohead_detailed(trained_3_1, test_loader_2h)

if 'results_group3_2h' not in globals():
    results_group3_2h = {}
if 'paths_group3_2h' not in globals():
    paths_group3_2h = {}

print(f"\n[NHOM3-1] AspectF1={a_f1:.4f} | GT-Sent={s_f1_gt:.4f} | Pipe-Sent={s_f1_pipe:.4f} | Combined={comb:.4f}")
print("Detail:", detail_3_1)
print("Saved:", path_3_1)
results_group3_2h["Shared_GatedFusion_Full"] = {"aspect_f1": a_f1, "sentiment_f1": s_f1_gt,
    "pipeline_sentiment_f1": s_f1_pipe, "combined": comb, "detail": detail_3_1 }



=== NHÓM 1: gated_fusion | FullLoss | LoRA(r=32,a=64,ffn) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LoRA | r=32, alpha=64, use_ffn=True
trainable params: 5,357,568 || all params: 140,355,840 || trainable%: 3.8171
[Scheduler] Linear Warmup+Decay enabled | warmup=600/7500 steps


TwoHead_Shared_GatedFusion | Epoch 1:   0%|          | 0/250 [00:00<?, ?it/s]

...

Eval twohead:   0%|          | 0/32 [00:00<?, ?it/s]

TwoHead_Shared_GatedFusion | Epoch 24 | Loss=0.0008 | Aspect=0.9807 GT-Sent=0.9474 Pipe-Sent=0.9560 Comb=0.9684 | Best val_pipeline_sent_f1=0.9551 @ep19

>>> Early stopping triggered at epoch 24.
>>> Best val_pipeline_sent_f1 = 0.9551 at epoch 19

[Best Checkpoint] Loading BEST weights into returned model from: ./models/TwoHead_Shared_GatedFusion_BEST_20260613_200840
Saved TwoHead LoRA model (shared mode=gated_fusion) to ./models/TwoHead_Shared_GatedFusion_LAST_20260613_201336
Final (last) model also saved at: ./models/TwoHead_Shared_GatedFusion_LAST_20260613_201336

Best val_pipeline_sent_f1 achieved: 0.9551 at epoch 19
Returning model from: ./models/TwoHead_Shared_GatedFusion_BEST_20260613_200840



Eval twohead:   0%|          | 0/32 [00:00<?, ?it/s]

Evaluating twohead detailed:   0%|          | 0/32 [00:00<?, ?it/s]


===== TWOHEAD DETAILED EVAL (with PIPELINE) =====
Overall Mention F1                  : 0.9712
Overall Sentiment F1 (GT-gated)     : 0.9311
Overall Sentiment F1 (PIPELINE)     : 0.9539  <-- what users see
Per Aspect (mention | GT-sent | PIPELINE-sent):
  RoomQuality : 0.9744 | 0.9091 | 0.9335
  Noise       : 0.9798 | 0.9327 | 0.9464
  Wifi        : 0.9835 | 0.9188 | 0.9306
  Utilities   : 0.9659 | 0.9384 | 0.9661
  Parking     : 0.9761 | 0.9523 | 0.9730
  Security    : 0.9793 | 0.9114 | 0.9286
  Environment : 0.9630 | 0.9349 | 0.9677
  Landlord    : 0.9759 | 0.9377 | 0.9600
  Location    : 0.9582 | 0.9163 | 0.9508
  Price       : 0.9565 | 0.9586 | 0.9774

[NHOM3-1] AspectF1=0.9712 | GT-Sent=0.9311 | Pipe-Sent=0.9539 | Combined=0.9626
Detail: {'overall_mention_f1': 0.9712, 'overall_sentiment_f1': 0.9311, 'overall_pipeline_sentiment_f1': 0.9539, 'macro_per_aspect_mention_f1': 0.9712, 'macro_per_aspect_sentiment_f1': 0.931, 'macro_per_aspect_pipeline_sentiment_f1': 0.9534, 'per_aspect': 

## ConditionalAttention


In [ ]:
print("=== NHÓM 2: conditional_attention | FullLoss | LoRA(r=32,a=64,ffn) ===")

loss_fn_3 = ABSATwoHeadLoss(
    num_aspects=NUM_ASPECTS,
    use_gated=True,
    use_weighted=True,
    use_focal=True,
    gamma=2.0,
    aspect_weight=1.2,
    sentiment_weight=0.85
)

model_3_2 = LoRABERTABSA_TwoHead(
    model_name=MODEL_NAME,
    feature_mode="conditional_attention",
    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    use_ffn=True,
    dropout=0.1
).to(DEVICE)

trained_3_2, hist_3_2, path_3_2 = train_twohead_model(
    model=model_3_2,
    train_loader=train_loader_2h,
    val_loader=val_loader_2h,
    epochs=30,
    lr=2e-4,
    model_name_for_log="TwoHead_Shared_CondAttn",
    loss_fn=loss_fn_3,
    save_dir="./models",
    patience=5,
    monitor="val_pipeline_sent_f1",
    mode="max",
    min_delta=0.0015,
    warmup_ratio=0.08,
    use_scheduler=True,
    save_best=True
)

a_f1, s_f1_gt, s_f1_pipe, comb = evaluate_twohead(trained_3_2, test_loader_2h)
detail_3_2 = evaluate_twohead_detailed(trained_3_2, test_loader_2h)


print(f"\n[NHOM3-2] AspectF1={a_f1:.4f} | GT-Sent={s_f1_gt:.4f} | Pipe-Sent={s_f1_pipe:.4f} | Combined={comb:.4f}")
print("Detail:", detail_3_2)
print("Saved:", path_3_2)
results_group3_2h["Shared_CondAttn_Full"] = {"aspect_f1": a_f1, "sentiment_f1": s_f1_gt,
    "pipeline_sentiment_f1": s_f1_pipe, "combined": comb, "detail": detail_3_2 }



=== NHÓM 2: conditional_attention | FullLoss | LoRA(r=32,a=64,ffn) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LoRA | r=32, alpha=64, use_ffn=True
trainable params: 5,357,568 || all params: 140,355,840 || trainable%: 3.8171
[Scheduler] Linear Warmup+Decay enabled | warmup=600/7500 steps


TwoHead_Shared_CondAttn | Epoch 1:   0%|          | 0/250 [00:00<?, ?it/s]

...

LoRA | r=32, alpha=64, use_ffn=True
trainable params: 5,357,568 || all params: 140,355,840 || trainable%: 3.8171
Model loaded successfully and set to eval mode.
Saved TwoHead LoRA model (shared mode=conditional_attention) to ./models/TwoHead_Shared_CondAttn_LAST_20260613_204010
Final (last) model also saved at: ./models/TwoHead_Shared_CondAttn_LAST_20260613_204010

Best val_pipeline_sent_f1 achieved: 0.0000 at epoch 1
Returning model from: ./models/TwoHead_Shared_CondAttn_BEST_20260613_203506



Eval twohead:   0%|          | 0/32 [00:00<?, ?it/s]

Evaluating twohead detailed:   0%|          | 0/32 [00:00<?, ?it/s]


===== TWOHEAD DETAILED EVAL (with PIPELINE) =====
Overall Mention F1                  : 0.0000
Overall Sentiment F1 (GT-gated)     : 0.0000
Overall Sentiment F1 (PIPELINE)     : 0.0000  <-- what users see
Per Aspect (mention | GT-sent | PIPELINE-sent):
  RoomQuality : 0.0000 | 0.0000 | 0.0000
  Noise       : 0.0000 | 0.0000 | 0.0000
  Wifi        : 0.0000 | 0.0000 | 0.0000
  Utilities   : 0.0000 | 0.0000 | 0.0000
  Parking     : 0.0000 | 0.0000 | 0.0000
  Security    : 0.0000 | 0.0000 | 0.0000
  Environment : 0.0000 | 0.0000 | 0.0000
  Landlord    : 0.0000 | 0.0000 | 0.0000
  Location    : 0.0000 | 0.0000 | 0.0000
  Price       : 0.0000 | 0.0000 | 0.0000

[NHOM3-2] AspectF1=0.0000 | GT-Sent=0.0000 | Pipe-Sent=0.0000 | Combined=0.0000
Detail: {'overall_mention_f1': 0.0, 'overall_sentiment_f1': 0.0, 'overall_pipeline_sentiment_f1': 0.0, 'macro_per_aspect_mention_f1': 0.0, 'macro_per_aspect_sentiment_f1': nan, 'macro_per_aspect_pipeline_sentiment_f1': nan, 'per_aspect': {'RoomQuality': {

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## AttentionPooling

In [20]:
print("=== NHÓM 3: attention_pooling | FullLoss + higher r | ===")

loss_fn_3 = ABSATwoHeadLoss(
    num_aspects=NUM_ASPECTS,
    use_gated=True,
    use_weighted=True,
    use_focal=True,
    gamma=2.2,
    aspect_weight=1.0,
    sentiment_weight=0.95
)

model_3_3 = LoRABERTABSA_TwoHead(
    model_name=MODEL_NAME,
    feature_mode="attention_pooling",
    lora_r=40,
    lora_alpha=80,
    lora_dropout=0.05,
    use_ffn=True,
    dropout=0.1
).to(DEVICE)

trained_3_3, hist_3_3, path_3_3 = train_twohead_model(
    model=model_3_3,
    train_loader=train_loader_2h,
    val_loader=val_loader_2h,
    epochs=30,
    lr=2e-4,
    model_name_for_log="TwoHead_Shared_AttnPool_HiLoRA",
    loss_fn=loss_fn_3,
    save_dir="./models",
    patience=5,
    monitor="val_pipeline_sent_f1",
    mode="max",
    min_delta=0.0015,
    warmup_ratio=0.08,
    use_scheduler=True,
    save_best=True
)

a_f1, s_f1_gt, s_f1_pipe, comb = evaluate_twohead(trained_3_3, test_loader_2h)
detail_3_3 = evaluate_twohead_detailed(trained_3_3, test_loader_2h)


print(f"\n[NHOM3-3] AspectF1={a_f1:.4f} | GT-Sent={s_f1_gt:.4f} | Pipe-Sent={s_f1_pipe:.4f} | Combined={comb:.4f}")
print("Detail:", detail_3_3)
print("Saved:", path_3_3)

# Tóm tắt nhanh Nhóm 3 (2-head)
print("\n===== NHÓM 3 (2-HEAD) SUMMARY =====")
for k, v in results_group3_2h.items():
    print(f"{k}: aspect={v['aspect_f1']:.4f} sent={v['sentiment_f1']:.4f} comb={v['combined']:.4f}")
results_group3_2h["Shared_AttnPool_HiLoRA"] = {"aspect_f1": a_f1, "sentiment_f1": s_f1_gt,
    "pipeline_sentiment_f1": s_f1_pipe, "combined": comb, "detail": detail_3_3 }



=== NHÓM 3: attention_pooling | FullLoss + higher r | ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LoRA | r=40, alpha=80, use_ffn=True
trainable params: 6,696,960 || all params: 141,695,232 || trainable%: 4.7263
[Scheduler] Linear Warmup+Decay enabled | warmup=600/7500 steps


TwoHead_Shared_AttnPool_HiLoRA | Epoch 1:   0%|          | 0/250 [00:00<?, ?it/s]

...

LoRA | r=32, alpha=64, use_ffn=True
trainable params: 5,357,568 || all params: 140,355,840 || trainable%: 3.8171
Model loaded successfully and set to eval mode.
Saved TwoHead LoRA model (shared mode=attention_pooling) to ./models/TwoHead_Shared_AttnPool_HiLoRA_LAST_20260613_210649
Final (last) model also saved at: ./models/TwoHead_Shared_AttnPool_HiLoRA_LAST_20260613_210649

Best val_pipeline_sent_f1 achieved: 0.9536 at epoch 14
Returning model from: ./models/TwoHead_Shared_AttnPool_HiLoRA_BEST_20260613_210142



Eval twohead:   0%|          | 0/32 [00:00<?, ?it/s]

Evaluating twohead detailed:   0%|          | 0/32 [00:00<?, ?it/s]


===== TWOHEAD DETAILED EVAL (with PIPELINE) =====
Overall Mention F1                  : 0.9701
Overall Sentiment F1 (GT-gated)     : 0.9246
Overall Sentiment F1 (PIPELINE)     : 0.9437  <-- what users see
Per Aspect (mention | GT-sent | PIPELINE-sent):
  RoomQuality : 0.9658 | 0.9181 | 0.9317
  Noise       : 0.9756 | 0.9454 | 0.9593
  Wifi        : 0.9877 | 0.9250 | 0.9321
  Utilities   : 0.9706 | 0.9336 | 0.9604
  Parking     : 0.9761 | 0.9205 | 0.9368
  Security    : 0.9793 | 0.9154 | 0.9361
  Environment : 0.9630 | 0.9167 | 0.9377
  Landlord    : 0.9759 | 0.9094 | 0.9311
  Location    : 0.9470 | 0.9174 | 0.9388
  Price       : 0.9603 | 0.9429 | 0.9678

[NHOM3-3] AspectF1=0.9701 | GT-Sent=0.9246 | Pipe-Sent=0.9437 | Combined=0.9569
Detail: {'overall_mention_f1': 0.9701, 'overall_sentiment_f1': 0.9246, 'overall_pipeline_sentiment_f1': 0.9437, 'macro_per_aspect_mention_f1': 0.9701, 'macro_per_aspect_sentiment_f1': 0.9244, 'macro_per_aspect_pipeline_sentiment_f1': 0.9432, 'per_aspect':

NameError: name 'results_group3_2h' is not defined

## Inference ví dụ với 2-Head model

In [21]:
def predict_twohead(model, texts, aspects=ASPECTS, threshold=0.5):
    model.eval()
    results = []
    with torch.no_grad():
        for text in texts:
            tokens = tokenizer(text, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
            input_ids = tokens["input_ids"].to(DEVICE)
            amask = tokens["attention_mask"].to(DEVICE)
            a_logits, s_logits = model(input_ids, amask)
            a_probs = torch.sigmoid(a_logits).cpu().numpy()[0]
            s_probs = torch.sigmoid(s_logits).cpu().numpy()[0]
            rec = {}
            for i, asp in enumerate(aspects):
                mentioned = bool(a_probs[i] > threshold)
                rec[f"{asp}_mentioned"] = mentioned
                if mentioned:
                    st = i*3
                    # take the highest sentiment
                    cands = s_probs[st:st+3]
                    sent_idx = int(cands.argmax())
                    sent_label = [1, -1, 0][sent_idx]
                    rec[f"{asp}_sentiment"] = sent_label
                    rec[f"{asp}_probs"] = {"pos": float(cands[0]), "neg": float(cands[1]), "neu": float(cands[2])}
                else:
                    rec[f"{asp}_sentiment"] = None
            results.append(rec)
    return results

# Load mô hình từ ổ đĩa & Đánh giá chi tiết F1 (re-evaluate)

Hàm này giúp bạn load lại model 2-head đã train (từ `paths_group3_2h`) và tính lại các chỉ số F1 chi tiết:
- **Mention F1** (aspect detection): toàn cục + từng aspect
- **Sentiment F1** (chỉ tính trên sample có mention): toàn cục + từng aspect
- **F1 per aspect**: mention + sentiment cho từng khía cạnh riêng lẻ


In [18]:
from peft import PeftModel
import os

def load_twohead_model(save_path: str):
    """Load 2-head model từ thư mục đã lưu bởi save_twohead_model.
    Yêu cầu trong save_path có:
      - lora_adapter/ (thư mục từ peft)
      - aspect_head.pt
      - sentiment_head.pt
      - config.json (chứa feature_mode, num_aspects)
    """
    config_path = os.path.join(save_path, "config.json")
    lora_adapter_path = os.path.join(save_path, "lora_adapter")
    aspect_head_path = os.path.join(save_path, "aspect_head.pt")
    sentiment_head_path = os.path.join(save_path, "sentiment_head.pt")

    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Không tìm thấy config.json tại {save_path}")

    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    feature_mode = cfg.get("feature_mode", "gated_fusion")
    num_aspects = cfg.get("num_aspects", NUM_ASPECTS)

    print(f"Loading model from: {save_path}")
    print(f"  feature_mode = {feature_mode}")
    print(f"  num_aspects = {num_aspects}")

    # Load base encoder + LoRA adapter
    base_encoder = AutoModel.from_pretrained(MODEL_NAME)
    encoder = PeftModel.from_pretrained(base_encoder, lora_adapter_path)

    # Tạo wrapper với đúng feature_mode
    model = LoRABERTABSA_TwoHead(
        model_name=MODEL_NAME,
        feature_mode=feature_mode,
        num_aspects=num_aspects
    )
    model.encoder = encoder   # thay thế bằng peft model đã load

    # Load hai head
    model.aspect_head.load_state_dict(
        torch.load(aspect_head_path, map_location=DEVICE)
    )
    model.sentiment_head.load_state_dict(
        torch.load(sentiment_head_path, map_location=DEVICE)
    )

    model = model.to(DEVICE).eval()
    print("Model loaded successfully and set to eval mode.")
    return model


@torch.no_grad()
def evaluate_fresh(model, data_loader, aspects=ASPECTS, verbose=False):
    model.eval()
    all_aspect_true, all_aspect_pred = [], []
    per_aspect_sent_true = {asp: [] for asp in aspects}
    per_aspect_sent_pred = {asp: [] for asp in aspects}
    per_aspect_pipe_true = {asp: [] for asp in aspects}
    per_aspect_pipe_pred = {asp: [] for asp in aspects}

    for batch in data_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        a_labels = batch["aspect_labels"].cpu().numpy()
        s_labels = batch["sentiment_labels"].cpu().numpy()

        a_logits, s_logits = model(input_ids, attention_mask)
        a_pred = (torch.sigmoid(a_logits) > 0.5).int().cpu().numpy()
        s_pred = (torch.sigmoid(s_logits) > 0.5).int().cpu().numpy()

        all_aspect_true.append(a_labels)
        all_aspect_pred.append(a_pred)

        for b in range(a_labels.shape[0]):
            for i, asp in enumerate(aspects):
                st = i * 3
                # GT-gated
                if a_labels[b, i] > 0.5:
                    per_aspect_sent_true[asp].append(s_labels[b, st:st+3])
                    per_aspect_sent_pred[asp].append(s_pred[b, st:st+3])
                # Pipeline (pred-gated)
                if a_pred[b, i] > 0.5:
                    per_aspect_pipe_true[asp].append(s_labels[b, st:st+3])
                    per_aspect_pipe_pred[asp].append(s_pred[b, st:st+3])

    # Overall Mention F1
    a_true = np.concatenate(all_aspect_true, axis=0)
    a_pred = np.concatenate(all_aspect_pred, axis=0)
    overall_mention_f1 = f1_score(a_true, a_pred, average="macro", zero_division=0)

    # Overall Sentiment GT-Gated F1
    all_sent_true = [np.stack(per_aspect_sent_true[asp]) for asp in aspects if len(per_aspect_sent_true[asp]) > 0]
    all_sent_pred = [np.stack(per_aspect_sent_pred[asp]) for asp in aspects if len(per_aspect_sent_pred[asp]) > 0]
    overall_sentiment_f1 = f1_score(np.concatenate(all_sent_true), np.concatenate(all_sent_pred), average="macro", zero_division=0) if all_sent_true else 0.0

    # Overall Pipeline F1
    all_pipe_true = [np.stack(per_aspect_pipe_true[asp]) for asp in aspects if len(per_aspect_pipe_true[asp]) > 0]
    all_pipe_pred = [np.stack(per_aspect_pipe_pred[asp]) for asp in aspects if len(per_aspect_pipe_pred[asp]) > 0]
    overall_pipeline_f1 = f1_score(np.concatenate(all_pipe_true), np.concatenate(all_pipe_pred), average="macro", zero_division=0) if all_pipe_true else 0.0

    # Per-aspect metrics
    per_aspect = {}
    for i, asp in enumerate(aspects):
        m_f1 = f1_score(a_true[:, i], a_pred[:, i], average="binary", zero_division=0)
        s_f1 = f1_score(np.stack(per_aspect_sent_true[asp]), np.stack(per_aspect_sent_pred[asp]), average="macro", zero_division=0) if len(per_aspect_sent_true[asp]) > 0 else 0.0
        p_f1 = f1_score(np.stack(per_aspect_pipe_true[asp]), np.stack(per_aspect_pipe_pred[asp]), average="macro", zero_division=0) if len(per_aspect_pipe_true[asp]) > 0 else 0.0

        per_aspect[asp] = {"mention_f1": round(m_f1, 4), "sentiment_f1": round(s_f1, 4), "pipeline_sentiment_f1": round(p_f1, 4)}

    return {
        "overall_mention_f1": overall_mention_f1,
        "overall_sentiment_f1": overall_sentiment_f1,
        "overall_pipeline_sentiment_f1": overall_pipeline_f1,
        "per_aspect": per_aspect
    }